# 05 - Recover expired work leases

Runs every five minutes. Recovery first fences stale work as `RECOVERING`, then upserts the attempt history, then releases the work to `RETRY_WAIT` or `DEAD_LETTERED`. Every phase is retry-safe, including a claim that committed before its attempt row was created.

**After importing into Fabric:** On the configuration code cell, select **... -> Toggle parameter cell** and confirm the parameter indicator. Then attach and pin `people_counter_<environment>` as this notebook's default Lakehouse.

In [ ]:
DATABASE = ""
TABLE_PREFIX = "people_counter"
EXPIRY_GRACE_MINUTES = 5
HEARTBEAT_TIMEOUT_MINUTES = 20
MAX_RECOVERIES_PER_RUN = 1000

In [ ]:
from datetime import datetime, timedelta, timezone
import json
import re

import notebookutils
from delta.tables import DeltaTable
from pyspark.sql import SparkSession, functions as F


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
database = DATABASE.strip()
prefix = TABLE_PREFIX.strip()
if database and IDENTIFIER.fullmatch(database) is None:
    raise ValueError("DATABASE is not a valid identifier")
if IDENTIFIER.fullmatch(prefix) is None:
    raise ValueError("TABLE_PREFIX is not a valid identifier")
grace_minutes = int(EXPIRY_GRACE_MINUTES)
heartbeat_timeout_minutes = int(HEARTBEAT_TIMEOUT_MINUTES)
limit = int(MAX_RECOVERIES_PER_RUN)
if grace_minutes < 0 or heartbeat_timeout_minutes < 1 or limit < 1:
    raise ValueError("Recovery parameters are invalid")


def table(suffix: str) -> str:
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
work_table = table("video_work")
attempts_table = table("video_attempts")
now = datetime.now(timezone.utc)
lease_cutoff = now - timedelta(minutes=grace_minutes)
heartbeat_cutoff = now - timedelta(minutes=heartbeat_timeout_minutes)
active_states = ["LEASED", "STAGING", "RUNNING", "WRITING"]

candidates = (
    spark_session.table(work_table)
    .where(
        (
            F.col("status").isin(active_states)
            & (
                (F.col("lease_expires_at") < F.lit(lease_cutoff))
                | (F.col("last_heartbeat_at") < F.lit(heartbeat_cutoff))
            )
        )
        | (F.col("status") == "RECOVERING")
    )
    .where(F.col("committed_attempt_id").isNull() & F.col("lease_owner_attempt_id").isNotNull())
    .orderBy("lease_expires_at")
    .limit(limit)
    .select(
        "work_id",
        F.col("lease_owner_attempt_id").alias("attempt_id"),
        F.col("lease_dispatcher_id").alias("dispatcher_id"),
        "attempt_count",
        "max_attempts",
        "lease_acquired_at",
        "last_heartbeat_at",
        "config_sha256",
        "capture_date",
    )
    .withColumn(
        "next_status",
        F.when(F.col("attempt_count") >= F.col("max_attempts"), F.lit("DEAD_LETTERED")).otherwise(F.lit("RETRY_WAIT")),
    )
    .withColumn("not_before_at", F.expr("current_timestamp() + INTERVAL 5 MINUTES"))
    .withColumn("lease_cutoff", F.lit(lease_cutoff))
    .withColumn("heartbeat_cutoff", F.lit(heartbeat_cutoff))
    .cache()
)
candidate_count = candidates.count()

if candidate_count:
    (
        DeltaTable.forName(spark_session, work_table)
        .alias("t")
        .merge(
            candidates.alias("s"),
            (
                "t.work_id = s.work_id AND t.capture_date = s.capture_date AND "
                "t.lease_owner_attempt_id = s.attempt_id"
            ),
        )
        .whenMatchedUpdate(
            condition=(
                "t.committed_attempt_id IS NULL AND ("
                "t.status = 'RECOVERING' OR ("
                "t.status IN ('LEASED', 'STAGING', 'RUNNING', 'WRITING') AND "
                "(t.lease_expires_at < s.lease_cutoff OR t.last_heartbeat_at < s.heartbeat_cutoff)))"
            ),
            set={
                "status": "'RECOVERING'",
                "last_error_category": "'WATCHDOG'",
                "last_error_type": "'LeaseExpired'",
                "last_error_message": "'Worker heartbeat or lease expired before completion'",
            },
        )
        .execute()
    )

fenced = (
    candidates.alias("s")
    .join(
        spark_session.table(work_table).alias("w"),
        (F.col("s.work_id") == F.col("w.work_id"))
        & (F.col("s.capture_date") == F.col("w.capture_date"))
        & (F.col("s.attempt_id") == F.col("w.lease_owner_attempt_id")),
        "inner",
    )
    .where((F.col("w.status") == "RECOVERING") & F.col("w.committed_attempt_id").isNull())
    .select("s.*")
    .cache()
)
fenced_count = fenced.count()
requeued_count = fenced.where(F.col("next_status") == "RETRY_WAIT").count()
dead_lettered_count = fenced.where(F.col("next_status") == "DEAD_LETTERED").count()

if fenced_count:
    attempt_source = fenced.select(
        "attempt_id",
        "work_id",
        F.coalesce("dispatcher_id", F.lit("watchdog-recovered-claim")).alias("dispatcher_id"),
        F.lit(None).cast("string").alias("pipeline_run_id"),
        F.lit(None).cast("string").alias("activity_run_id"),
        F.lit(None).cast("string").alias("fabric_job_instance_id"),
        F.lit(None).cast("string").alias("worker_execution_id"),
        F.lit(None).cast("string").alias("sdk_version"),
        F.lit(None).cast("string").alias("bundle_manifest_sha256"),
        "config_sha256",
        F.col("next_status").alias("status"),
        F.coalesce("lease_acquired_at", F.lit(now)).alias("claimed_at"),
        F.lit(None).cast("timestamp").alias("staging_started_at"),
        F.lit(None).cast("timestamp").alias("inference_started_at"),
        F.lit(None).cast("timestamp").alias("writing_started_at"),
        F.lit(now).alias("completed_at"),
        "last_heartbeat_at",
        F.lit(None).cast("string").alias("input_sha256"),
        F.lit(None).cast("long").alias("source_size_bytes"),
        F.lit(None).cast("double").alias("source_duration_seconds"),
        F.lit(None).cast("double").alias("source_fps"),
        F.lit(None).cast("long").alias("total_source_frames"),
        F.lit(None).cast("long").alias("processed_frames"),
        F.lit(None).cast("double").alias("effective_sample_fps"),
        F.lit(None).cast("double").alias("processing_seconds"),
        F.lit(None).cast("long").alias("distinct_people"),
        F.lit(None).cast("long").alias("line_in_count"),
        F.lit(None).cast("long").alias("line_out_count"),
        (F.col("next_status") == "RETRY_WAIT").alias("retryable"),
        F.lit("WATCHDOG").alias("error_category"),
        F.lit("LeaseExpired").alias("error_type"),
        F.lit("Worker heartbeat or lease expired before completion").alias("error_message"),
        "capture_date",
    )
    (
        DeltaTable.forName(spark_session, attempts_table)
        .alias("t")
        .merge(
            attempt_source.alias("s"),
            "t.attempt_id = s.attempt_id AND t.capture_date = s.capture_date",
        )
        .whenMatchedUpdate(
            condition="t.status <> 'SUCCEEDED'",
            set={
                "status": "s.status",
                "completed_at": "s.completed_at",
                "retryable": "s.retryable",
                "error_category": "s.error_category",
                "error_type": "s.error_type",
                "error_message": "s.error_message",
            },
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    (
        DeltaTable.forName(spark_session, work_table)
        .alias("t")
        .merge(
            fenced.alias("s"),
            (
                "t.work_id = s.work_id AND t.capture_date = s.capture_date AND "
                "t.lease_owner_attempt_id = s.attempt_id"
            ),
        )
        .whenMatchedUpdate(
            condition="t.status = 'RECOVERING' AND t.committed_attempt_id IS NULL",
            set={
                "status": "s.next_status",
                "not_before_at": "CASE WHEN s.next_status = 'RETRY_WAIT' THEN s.not_before_at ELSE NULL END",
                "queue_entered_at": "CASE WHEN s.next_status = 'RETRY_WAIT' THEN current_timestamp() ELSE t.queue_entered_at END",
                "lease_owner_attempt_id": "NULL",
                "lease_dispatcher_id": "NULL",
                "lease_acquired_at": "NULL",
                "lease_expires_at": "NULL",
            },
        )
        .execute()
    )

outcome = {
    "checked_at": now.isoformat(),
    "expired_candidates": candidate_count,
    "fenced_attempts": fenced_count,
    "requeued": requeued_count,
    "dead_lettered": dead_lettered_count,
}
fenced.unpersist()
candidates.unpersist()
print(json.dumps(outcome, sort_keys=True))

In [ ]:
notebookutils.notebook.exit(json.dumps(outcome, sort_keys=True))